In [2]:
# load the csv data and check success rate
import os
import pandas as pd
from pymatgen.io.cif import CifParser
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

In [3]:
model_name = "qwen3_32b"
action_name = "insert_between_atoms_action_sg"

# ..\..\results\AtomWorld\deepseek_chat\insert_between_atoms_action_sg\20251116_141923\evaluation_results.csv
# get the datetime folder name
folder = f"../../results/AtomWorld/{model_name}/{action_name}/"
datetime_folders = os.listdir(folder)
# get the latest folder
latest_folder = sorted(datetime_folders)[-1]
csv_path = os.path.join(folder, latest_folder, "evaluation_results.csv")
df = pd.read_csv(csv_path)

In [4]:
# analyze the success rate based on space group
def get_space_group(cif_string):
    try:
        parser = CifParser.from_str(cif_string)
        structure = parser.get_structures(primitive=True)[0]
        sga = SpacegroupAnalyzer(structure)
        return sga.get_space_group_symbol()
    except:
        return None
    

df['space_group'] = df['input_cif'].apply(get_space_group)

C:\Users\taoyu\AppData\Local\Temp\ipykernel_38420\3618020888.py:5: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structure = parser.get_structures(primitive=True)[0]
d:\Codes\AtomWorld\.venv\Lib\site-packages\pymatgen\io\cif.py:1349: UserWarning: Issues encountered while parsing CIF: 18 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return self.parse_structures(*args, **kwargs)
d:\Codes\AtomWorld\.venv\Lib\site-packages\pymatgen\io\cif.py:1349: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return self.parse_structures(*args, **kwargs)
d:\Codes\AtomWorld\.venv\Lib\site-packages\pymatgen\io\cif.py:1349: UserWarning: Issues en

In [5]:
sg_success_counts = {}

for sg, group in df.groupby('space_group'):
    success = len(group)
    sg_success_counts[sg] = success / 10

sg_success_counts



{'C2/m': 0.9,
 'Cmmm': 1.0,
 'Fm-3m': 0.9,
 'Fmmm': 1.0,
 'I4/mmm': 1.0,
 'Im-3m': 0.8,
 'Immm': 0.9,
 'P-1': 0.8,
 'P2/m': 0.8,
 'P4/mmm': 0.9,
 'P6/mmm': 1.0,
 'Pm-3m': 0.9,
 'Pmmm': 0.9,
 'R-3m': 1.0}

In [6]:
bravias_names = {
    'Fm-3m': 'cF',  # Face-centered Cubic
    'Im-3m': 'cI',  # Body-centered Cubic
    'Pm-3m': 'cP',  # Simple Cubic
    'P6/mmm': 'hP',  # Hexagonal
    'R-3m': 'hR',  # Rhombohedral
    'I4/mmm': 'tI',  # Body-centered Tetragonal
    'P4/mmm': 'tP',  # Simple Tetragonal
    'Fmmm': 'oF',  # Face-centered Orthorhombic
    'Immm': 'oI',  # Body-centered Orthorhombic
    'Cmmm': 'oS',  # Base-centered Orthorhombic
    'Pmmm': 'oP',  # Simple Orthorhombic
    'C2/m': 'mS',  # Base-centered Monoclinic
    'P2/m': 'mP',  # Primitive Monoclinic
    'P-1': 'aP',  # Primitive Triclinic
}

# get the table bravias_names, success_rate
table = []
for sg, bravias in bravias_names.items():
    success_rate = sg_success_counts.get(sg, 0)
    table.append((bravias, success_rate))

dataframe = pd.DataFrame(table, columns=['Bravias Name', 'Success Rate'])
dataframe

,Bravias Name,Success Rate
0,cF,0.9
1,cI,0.8
2,cP,0.9
3,hP,1.0
4,hR,1.0
5,tI,1.0
6,tP,0.9
7,oF,1.0
8,oI,0.9
9,oS,1.0
